# Sanity checks for the FC↔SC experimental notebook

Standalone sanity tests that quickly stress-test interpretations of headline results from
[`model_overviews/crossmodal_pca_pls_closed_form_overview.ipynb`](model_overviews/crossmodal_pca_pls_closed_form_overview.ipynb).
Each check is self-contained: builds its own `Sim`, runs the test, prints a compact verdict.

Run any check independently — they don't depend on the main notebook's kernel state.


## Setup

Builds a minimal `Sim` (shuffle_seed=0, precomputed data). All sanity checks reuse the
`_base` it produces; building it once amortizes the data-loading cost.


In [12]:
import importlib
import sys
from pathlib import Path

# Ensure project root is on path (so `import main, models, data` work).
_REPO_ROOT = Path.cwd()
while _REPO_ROOT.name != "Conn2Conn" and _REPO_ROOT.parent != _REPO_ROOT:
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import main
import models.configs
import models.architectures
import models.eval.metrics as _metrics
importlib.reload(models.configs)
importlib.reload(models.architectures)
importlib.reload(main)
from main import Sim
print(f"repo root: {_REPO_ROOT}")


repo root: /scratch/ans9868/Conn2Conn


In [13]:
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA

# Match the main notebook's defaults.
PARCELLATION = "Glasser"
DATA_LOAD_MODE = "precomputed"

# Closed-form model config map (used to point Sim at the right yaml).
CLOSED_FORM_MODELS = {
    "CrossModalPCA": {"config": _REPO_ROOT / "models/configs/CrossModalPCA.yml"},
}

# Build Sim once. seed=0 = canonical split. direction = FC->SC to match the main notebook.
_sim = Sim(
    model_name="CrossModalPCA",
    config_path=str(CLOSED_FORM_MODELS["CrossModalPCA"]["config"]),
    source="FC", target="SC",
    parcellation=PARCELLATION, shuffle_seed=0,
    data_load_mode=DATA_LOAD_MODE,
)
_base = _sim.base
_train_idx = _base.trainvaltest_partition_indices["train"]
_test_idx  = _base.trainvaltest_partition_indices["test"]

print(f"train={len(_train_idx)}  test={len(_test_idx)}  parc={PARCELLATION}")
print(f"FS volume columns ({len(_base.fs_volume_columns)}):")
for i, c in enumerate(_base.fs_volume_columns):
    print(f"  [{i:>2}] {c}")


/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


train=683  test=195  parc=Glasser
FS volume columns (16):
  [ 0] FS_IntraCranial_Vol
  [ 1] FS_BrainSeg_Vol
  [ 2] FS_BrainSeg_Vol_No_Vent
  [ 3] FS_BrainSeg_Vol_No_Vent_Surf
  [ 4] FS_LCort_GM_Vol
  [ 5] FS_RCort_GM_Vol
  [ 6] FS_TotCort_GM_Vol
  [ 7] FS_SubCort_GM_Vol
  [ 8] FS_Total_GM_Vol
  [ 9] FS_SupraTentorial_Vol
  [10] FS_L_WM_Vol
  [11] FS_R_WM_Vol
  [12] FS_Tot_WM_Vol
  [13] FS_Mask_Vol
  [14] FS_BrainSegVol_eTIV_Ratio
  [15] FS_MaskVol_eTIV_Ratio


In [14]:
# Full 6-metric panel helper, matches the main notebook's _full_panel_eval exactly.
# Inlined here so this notebook stays self-contained.

def _full_panel_eval(y_pred, y_true, target_train_mean_vec):
    """Returns dict with mse, r2, pearson, demeaned_pearson, top1_acc, avg_rank.
    Matches the project conventions in models/eval/metrics.py."""
    yp = np.asarray(y_pred, dtype=np.float32)
    yt = np.asarray(y_true, dtype=np.float32)
    mu = np.asarray(target_train_mean_vec, dtype=np.float32)
    cc_raw = _metrics.compute_corr_matrix(
        torch.tensor(yt, dtype=torch.float32),
        torch.tensor(yp, dtype=torch.float32),
    ).cpu().numpy() if hasattr(_metrics.compute_corr_matrix, '__call__') else None
    # Fallback: use the same compute_corr_matrix the main notebook uses.
    cc_raw = _metrics.compute_corr_matrix(
        torch.tensor(yt, dtype=torch.float32),
        torch.tensor(yp, dtype=torch.float32),
    )
    if hasattr(cc_raw, "cpu"):
        cc_raw = cc_raw.cpu().numpy()
    panel = _metrics.compute_basic_regression_metrics(
        torch.tensor(yp, dtype=torch.float32),
        torch.tensor(yt, dtype=torch.float32),
        corr_matrix=torch.tensor(cc_raw, dtype=torch.float32),
        corr_matrix_demeaned=None,
    )
    # demeaned-r convention: cosine of (y - mu) per subject, then mean.
    yp_dm = yp - mu
    yt_dm = yt - mu
    num   = (yp_dm * yt_dm).sum(axis=1)
    den_p = np.sqrt((yp_dm ** 2).sum(axis=1))
    den_t = np.sqrt((yt_dm ** 2).sum(axis=1))
    panel = {k: (float(v) if hasattr(v, "item") else float(v)) for k, v in panel.items()} if isinstance(panel, dict) else dict(panel)
    panel["demeaned_pearson"] = float((num / (den_p * den_t + 1e-10)).mean())
    return panel

def _fmt_panel(p, label):
    return (f"{label:32s}  mse={p['mse']:.4f}  r2={p['r2']:+.4f}  "
            f"pearson={p['pearson']:.4f}  demeaned={p['demeaned_pearson']:.4f}  "
            f"top1={p['top1_acc']:.4f}  avg_rank={p['avg_rank']:.4f}")

print("Helpers defined: _full_panel_eval, _fmt_panel")


Helpers defined: _full_panel_eval, _fmt_panel


## Sanity check 1 — Is `bv → SC` really about brain *structure*, or just brain *size*?

### The question

In the main notebook, OLS with 16 FreeSurfer volume features predicts SC edges at
**demeaned-r ≈ 0.167** (10-seed mean from Exp 7). That's a high number for an
anatomy-only predictor, and it sits **above** several connectome-based baselines.
Two competing interpretations of where the 0.167 comes from:

**(a) Brain size as a single dimension.** People intuit "brain volume features" as
"how big the brain is", essentially one number. If that's right, then the 16 features
should be heavily redundant — they all just encode "this person has a big/small brain"
in different ways. The model would be exploiting a 1-dimensional brain-size signal.

**(b) Brain compositional pattern.** The 16 features encode much more than total size:
proportional composition (cortical vs subcortical GM, GM vs WM ratio), regional
distribution (left vs right, supratentorial vs infratentorial), ventricle-to-parenchyma
ratio, etc. Brain *composition* — not size — could be what's predicting SC, reflecting
shared neurodevelopmental factors that shape both tissue distribution and structural
connectivity.

### Why this matters for the writeup

If (a) is right, the framing is "brain size predicts SC at 0.17" — a slightly trivial
result with a clear univariate explanation.

If (b) is right, the framing is "anatomy *composition* predicts SC at 0.17" — a richer
finding about shared neurodevelopmental geometry. The bv+demo baseline that beats the
connectome on demeaned-r becomes substantively interesting, not just a brain-size
confound.

### The test (30 seconds of compute)

Fit a clean 1-feature OLS using only `FS_BrainSeg_Vol` (total brain volume, the most
direct "brain size" scalar). Compare its demeaned-r to the 16-feature OLS.

- If the 1-feature OLS gets demeaned-r ≈ 0.06–0.10 → the 16-feature 0.167 comes from
  compositional structure beyond size. Interpretation **(b)** validated.
- If the 1-feature OLS gets demeaned-r ≈ 0.14–0.17 → the result is essentially brain
  size. Interpretation **(a)** validated.
- Anywhere in between → mixed; total size contributes but composition adds non-trivial
  signal on top.

### Note on scale (why z-scoring isn't doing the work)

For multi-output OLS, predictions are scale-invariant — z-scoring inputs only changes the
fitted coefficients (rescaled to compensate), not the predicted values or any downstream
metric. So whether `_base.fs_volumes_z` is z-scored or raw doesn't affect this test's
result. We confirm that below as a third row.


### Pre-step: verify the actual preprocessing on the data

Before running the OLS comparison, make the z-scoring audit **empirical, not just
a code-comment claim**. The project asserts in `data/hcp_dataset.py:297-309` that
`_base.fs_volumes_z` is z-scored using **training-split statistics only**. Two things
to verify from the actual arrays in this notebook:

1. **Train columns** have mean ~ 0 and std ~ 1 (proves z-scoring *did happen*).
2. **Test columns** have mean ≠ 0 and std ≠ 1 (proves the z-scoring used **train-only**
   stats — otherwise the test set would also normalize to 0/1).

Both must hold. If either fails the rest of this sanity check (and any conclusion about
"brain size" or "composition") is meaningless because we'd be analyzing the wrong data.


In [15]:
# ----- Verify the underlying preprocessing on _base.fs_volumes_z -----
# This block produces an explicit audit so the next experiment is operating on data we
# have inspected, not on data we trust by reading data/hcp_dataset.py.

print("Z-score normalization vectors stored on _base (from HCP_Base.__init__):")
print(f"  _base.fs_vol_train_mean: shape={_base.fs_vol_train_mean.shape}  "
      f"dtype={_base.fs_vol_train_mean.dtype}")
print(f"  _base.fs_vol_train_std:  shape={_base.fs_vol_train_std.shape}   "
      f"dtype={_base.fs_vol_train_std.dtype}")
print()

# ---- Audit 1: training-set columns should be ~ (0, 1). ----
_train_mu = _base.fs_volumes_z[_train_idx].mean(axis=0)
_train_sd = _base.fs_volumes_z[_train_idx].std(axis=0)

print("(1) TRAIN-set z-scored stats (should be ~0 mean, ~1 std):")
for i in range(len(_base.fs_volume_columns)):
    print(f"  [{i:>2}] {_base.fs_volume_columns[i]:32s}"
          f"  train_mean = {_train_mu[i]:+.4e}    train_std = {_train_sd[i]:.6f}")
print(f"  -> max |train_mean|      = {abs(_train_mu).max():.4e}")
print(f"  -> max |train_std  - 1|  = {abs(_train_sd - 1).max():.4e}")
# Thresholds chosen for float32 precision: fs_volumes_z is float32 (~7-digit precision).
# Per-column mean accumulation across N=683 elements has noise on the order of sqrt(N)*eps_f32 ~ 3e-6,
# so a properly z-scored float32 array can show residual means up to ~1e-5 cleanly within precision.
# Threshold 1e-4 gives ~10x margin while still detecting a real un-z-scored array (which would have
# raw-scale means on the order of 1e6 for FS volumes in mm^3).
_pass_train = (abs(_train_mu).max() < 1e-4) and (abs(_train_sd - 1).max() < 1e-4)
print(f"  -> {'PASS' if _pass_train else 'FAIL'}: train cols are within tolerance of (mean=0, std=1).")
print()

# ---- Audit 2: test-set columns should be NEAR but NOT EXACTLY (0, 1). ----
_test_mu = _base.fs_volumes_z[_test_idx].mean(axis=0)
_test_sd = _base.fs_volumes_z[_test_idx].std(axis=0)

print("(2) TEST-set z-scored stats (should be near but NOT exactly 0/1 -- proves train-only):")
for i in range(len(_base.fs_volume_columns)):
    print(f"  [{i:>2}] {_base.fs_volume_columns[i]:32s}"
          f"  test_mean  = {_test_mu[i]:+.4f}    test_std  = {_test_sd[i]:.4f}")
print(f"  -> max |test_mean|       = {abs(_test_mu).max():.4f}")
print(f"  -> max |test_std  - 1|   = {abs(_test_sd - 1).max():.4f}")
# We expect non-zero but bounded — if test mean is exactly 0 something is suspicious.
_pass_test = (abs(_test_mu).max() > 1e-3) and (abs(_test_mu).max() < 1.0)
print(f"  -> {'PASS' if _pass_test else 'FAIL'}: test cols are non-zero/non-1 (z-scoring used TRAIN stats only).")
print()

print("=" * 78)
if _pass_train and _pass_test:
    print("Preprocessing audit PASSED. fs_volumes_z is z-scored on train-only stats,")
    print("as claimed in data/hcp_dataset.py:297-309. Sanity check 1 below is valid.")
else:
    print("Preprocessing audit FAILED. Do not trust the OLS results in the next cells")
    print("until the failure is understood. Check data/hcp_dataset.py and re-run setup.")
print("=" * 78)


Z-score normalization vectors stored on _base (from HCP_Base.__init__):
  _base.fs_vol_train_mean: shape=(16,)  dtype=float32
  _base.fs_vol_train_std:  shape=(16,)   dtype=float32

(1) TRAIN-set z-scored stats (should be ~0 mean, ~1 std):
  [ 0] FS_IntraCranial_Vol               train_mean = +1.1619e-06    train_std = 1.000000
  [ 1] FS_BrainSeg_Vol                   train_mean = +2.4246e-06    train_std = 1.000000
  [ 2] FS_BrainSeg_Vol_No_Vent           train_mean = -3.9891e-07    train_std = 1.000000
  [ 3] FS_BrainSeg_Vol_No_Vent_Surf      train_mean = -1.3336e-06    train_std = 1.000000
  [ 4] FS_LCort_GM_Vol                   train_mean = -8.2284e-06    train_std = 1.000000
  [ 5] FS_RCort_GM_Vol                   train_mean = +1.1846e-06    train_std = 1.000000
  [ 6] FS_TotCort_GM_Vol                 train_mean = -1.4473e-06    train_std = 1.000000
  [ 7] FS_SubCort_GM_Vol                 train_mean = +5.1236e-06    train_std = 1.000000
  [ 8] FS_Total_GM_Vol                  

In [16]:
# ----- Locate the single-feature column -----
_single_feature_name = "FS_BrainSeg_Vol"
_col_idx = list(_base.fs_volume_columns).index(_single_feature_name)
print(f"Using feature '{_single_feature_name}' at column index {_col_idx} of fs_volumes_z")

# ----- Build X matrices -----
# 16-feature (z-scored, as the main notebook uses):
X16_train = _base.fs_volumes_z[_train_idx]
X16_test  = _base.fs_volumes_z[_test_idx]
# 1-feature (z-scored, same column):
X1_train  = _base.fs_volumes_z[_train_idx, _col_idx:_col_idx+1]
X1_test   = _base.fs_volumes_z[_test_idx,  _col_idx:_col_idx+1]

# ----- Targets: raw SC edge vectors -----
Y_train = np.asarray(_base.sc_upper_triangles[_train_idx], dtype=np.float32)
Y_test  = np.asarray(_base.sc_upper_triangles[_test_idx],  dtype=np.float32)
SC_train_mean = Y_train.mean(axis=0)

print(f"X16_train {X16_train.shape}   X1_train {X1_train.shape}   Y_train {Y_train.shape}")

# ----- Fit and predict -----
_reg16 = LinearRegression().fit(X16_train, Y_train)
_reg1  = LinearRegression().fit(X1_train,  Y_train)
Y16_pred = _reg16.predict(X16_test).astype(np.float32)
Y1_pred  = _reg1.predict(X1_test ).astype(np.float32)

# ----- Score both -----
panel16 = _full_panel_eval(Y16_pred, Y_test, SC_train_mean)
panel1  = _full_panel_eval(Y1_pred,  Y_test, SC_train_mean)

print()
print(_fmt_panel(panel16, "16-feature OLS  (bv -> SC)"))
print(_fmt_panel(panel1,  f"1-feature OLS   ({_single_feature_name} -> SC)"))

# ----- Diagnostic ratios -----
print()
print("=== Comparison ===")
ratio_dm = panel16['demeaned_pearson'] / max(panel1['demeaned_pearson'], 1e-12)
print(f"  demeaned-r:  16-feat {panel16['demeaned_pearson']:.4f}  vs  1-feat {panel1['demeaned_pearson']:.4f}")
print(f"  ratio (16-feat / 1-feat) = {ratio_dm:.3f}x")
print(f"  delta (16-feat - 1-feat) = {panel16['demeaned_pearson'] - panel1['demeaned_pearson']:+.4f}")


Using feature 'FS_BrainSeg_Vol' at column index 1 of fs_volumes_z
X16_train (683, 16)   X1_train (683, 1)   Y_train (683, 64620)

16-feature OLS  (bv -> SC)        mse=0.0053  r2=+0.0093  pearson=0.9155  demeaned=0.1670  top1=0.1026  avg_rank=0.8789
1-feature OLS   (FS_BrainSeg_Vol -> SC)  mse=0.0053  r2=+0.0176  pearson=0.9158  demeaned=0.1476  top1=0.0205  avg_rank=0.7913

=== Comparison ===
  demeaned-r:  16-feat 0.1670  vs  1-feat 0.1476
  ratio (16-feat / 1-feat) = 1.132x
  delta (16-feat - 1-feat) = +0.0194


### Confirmation: OLS is scale-invariant (no z-score effect)

Quick check that fitting the 1-feature OLS on the *raw* (un-z-scored) brain volume gives
numerically identical predictions to the z-scored version above. This isolates the "is
z-scoring doing the work?" worry — answer is no, OLS predictions on the same column are
invariant to monotonic affine rescaling of the input.


In [17]:
# Pull the raw BrainSeg column (un-z-scored). We have the z-scored array on _base, but
# we can reverse the standardization to get raw using _base.fs_vol_train_mean / std at
# the same column index.
_mean_col = float(_base.fs_vol_train_mean[_col_idx])
_std_col  = float(_base.fs_vol_train_std[_col_idx])
X1_raw_train = _base.fs_volumes_z[_train_idx, _col_idx:_col_idx+1] * _std_col + _mean_col
X1_raw_test  = _base.fs_volumes_z[_test_idx,  _col_idx:_col_idx+1] * _std_col + _mean_col

_reg1_raw = LinearRegression().fit(X1_raw_train, Y_train)
Y1_raw_pred = _reg1_raw.predict(X1_raw_test).astype(np.float32)

# Compare predictions (should be identical to many decimal places).
_max_abs_diff = float(np.abs(Y1_pred - Y1_raw_pred).max())
print(f"max |Y1_pred (z-scored) - Y1_pred (raw)| = {_max_abs_diff:.2e}")
print(f"  -> {'identical (as expected, OLS is scale-invariant)' if _max_abs_diff < 1e-3 else 'DIFFERENT (unexpected, investigate)'}")

# Also report the panel for the raw-input version as belt-and-suspenders.
panel1_raw = _full_panel_eval(Y1_raw_pred, Y_test, SC_train_mean)
print()
print(_fmt_panel(panel1,     f"1-feature OLS   (z-scored {_single_feature_name})"))
print(_fmt_panel(panel1_raw, f"1-feature OLS   (raw      {_single_feature_name})"))


max |Y1_pred (z-scored) - Y1_pred (raw)| = 4.77e-07
  -> identical (as expected, OLS is scale-invariant)

1-feature OLS   (z-scored FS_BrainSeg_Vol)  mse=0.0053  r2=+0.0176  pearson=0.9158  demeaned=0.1476  top1=0.0205  avg_rank=0.7913
1-feature OLS   (raw      FS_BrainSeg_Vol)  mse=0.0053  r2=+0.0176  pearson=0.9158  demeaned=0.1476  top1=0.0205  avg_rank=0.7913


### How to read the result

Inspect the printed `demeaned_pearson` for the 1-feature row, then refer back to the
decision rule:

| 1-feature demeaned-r | Verdict |
|---|---|
| ≤ 0.10 | **Compositional**. The 0.167 comes from the relationship structure across the 16 volumes — proportional composition of cortical / subcortical / WM / ventricle, etc. The bv→SC baseline really is "anatomy structure predicts SC", not "brain size predicts SC". Write up the bv+demo baseline as anatomy-driven structure, not size confound. |
| 0.10 – 0.14 | **Mixed**. Total brain size carries about half the predictive signal of the 16-feature version; the other half is compositional. Both framings have some merit. |
| ≥ 0.14 | **Brain size dominates**. The 16-feature OLS is mostly extracting a 1D brain-size signal in 16 redundant flavors. The bv+demo baseline becomes a "brain size" confound, less substantively interesting than the compositional reading. |

Whatever number lands, also note:
- `top1_acc` and `avg_rank` for the 1-feature OLS. A single brain-size feature shouldn't
  produce strong identifiability — if it does (top1 ≫ chance, avg_rank ≫ 0.5), that's
  another signal worth interpreting.
- The z-scored vs raw confirmation should be essentially zero difference (< 1e-3).
  Any larger difference is a bug.


### Result + interpretation (recorded 2026-05-28)

**Preprocessing audit (cell 7):** ✅ PASSED.

| Audit | Worst-case | Verdict |
|---|---|---|
| TRAIN-set z-scoring (mean → 0, std → 1) | `max\|mean\| = 1.86e-5`, `max\|std − 1\| = 6.6e-7` | PASS — at float32 precision floor |
| TEST-set NOT re-normalized (proves train-only) | `max\|mean\| = 0.0464`, `max\|std − 1\| = 0.237` | PASS — non-zero/non-1 as expected |

Audit values confirm the project's claim in `data/hcp_dataset.py:297-309`: `fs_volumes_z`
is z-scored using **train-only statistics**. The OLS comparison below is therefore
operating on the data we have inspected, not just the data we trust by reading
docstrings.

---


**Numbers (seed 0, n_test = 195):**

| Metric          | 16-feature bv | 1-feature `FS_BrainSeg_Vol` | Delta |
|-----------------|---------------|-----------------------------|-------|
| demeaned-r      | 0.1670        | **0.1476**                  | +0.019 (1.13×) |
| top1_acc        | 0.1026        | **0.0205**                  | −0.082 (**5.0× drop**) |
| avg_rank        | 0.8789        | 0.7913                      | −0.088 |
| pearson         | 0.9155        | 0.9158                      | ≈ 0 |
| r²              | +0.0093       | +0.0176                     | 1-feat *higher* on test |
| mse             | 0.0053        | 0.0053                      | ≈ 0 |

**Scale-invariance sub-check:** `max |Y1_pred_zscored − Y1_pred_raw| = 4.77e-07`. Numerically
identical — confirms OLS predictions are invariant to z-scoring on the same column. The
z-score-vs-raw worry is fully retired.

**Verdict — neither (a) nor (b) pure; both, on different metrics:**

- **On demeaned-r** → **(a) brain size dominates.** 1-feature OLS captures **88 % of the
  16-feature signal** (0.148 / 0.167). Edge-value prediction is essentially "scale the
  per-edge train mean by total brain volume." Above the originally-stated decision
  threshold of 0.14 → "brain size dominates" range.
- **On identifiability** → **(b) composition dominates.** Collapsing 16 → 1 feature
  drops top1 from 0.103 → 0.021 (5.0× drop) and avg_rank from 0.879 → 0.791. Brain
  size alone cannot fingerprint subjects; the remaining 15 compositional features
  carry the inter-subject distinguishing signal almost entirely.

**Mechanistic reading:** brain size shifts the mean prediction vector into the right
neighborhood (good for `demeaned_pearson`, which is a per-subject cosine of (y − mu)).
But distinguishing which subject is whom needs compositional channels (good for `top1`
and `avg_rank`, which are gallery-rank measures).

**Tertiary observation:** 1-feature test r² (0.018) > 16-feature test r² (0.009). The
16-feature OLS is slightly noise-fit on per-edge variance — adding 15 features to a
brain-size-dominated signal hurts per-edge variance accuracy marginally, while still
gaining identifiability. Doesn't affect the headline (demeaned-r is the metric of
interest) but consistent with the "size dominates edges, composition dominates
identifiability" picture.

**Writeup framing recommendation:**

> "Anatomy predicts SC at demeaned-r = 0.167, of which ~88 % is recoverable from
> total brain volume alone (1-feature OLS: 0.148). The remaining ~12 % comes from
> compositional structure across the 16 FreeSurfer volumes. Compositional structure
> contributes disproportionately to **identifiability**: a single-scalar brain-size
> predictor achieves top1 = 0.021 (~4× chance), while the 16-feature version
> achieves top1 = 0.103 (~20× chance) — a 5× lift driven by composition, not size."

This is the cleanest framing for the bv → SC baseline. It replaces both the naive
"brain size confound" and the naive "anatomy structure" framings with the
metric-aware version.

**Prior update:** I (and the user) expected the 1-feature demeaned-r at ~0.06–0.10.
Actual was 0.148 — substantially higher than priors. The lesson: "individual deviation
from the group connectome mean" is largely "individual deviation from the group brain
size", at least in the demeaned-r sense. Future priors on bv-baseline experiments
should weight univariate brain-size effects more heavily on edge-value metrics.


---

## Future sanity checks (placeholders — add cells as needed)

- **Demeaned-r convention double-check.** Construct a residual prediction with known
  row-mean ≠ 0 and verify project's `compute_demeaned_pearson_r` matches a hand-rolled
  cosine-of-deviation formula. The bug from 2026-05-26 (cell 20 of the main notebook)
  motivated this.

- **Family-aware split: ratio of MZ pairs co-located in train vs val vs test.**
  Confirm `generate_train_val_test` keeps families together so no subject's twin is
  in another partition (would inflate test-set identifiability).

- **PCA basis stability across seeds.** Compare top-K loadings of `population_mean_pca`
  fit on seed 0 train vs seed 1 train. Cosine similarity of leading components — should
  be high (~0.95+) for low-K, drop for tail components.

- **Bootstrap CI stability vs n_boot.** Run the family-structure AUC bootstrap at
  n_boot ∈ {500, 1000, 2000, 5000} and confirm CIs are stable to ±0.005. Cheap; nice
  reviewer-defense.

- **Cognitive variables: distribution + missingness per seed.** Verify
  `CogTotalComp_Unadj` distribution doesn't have unexpected outliers; missingness rate
  per train/test split is stable across seeds.
